In [21]:
import pandas as pd
import numpy as np
from pathlib import Path
from pandas.tseries.offsets import MonthEnd

In [6]:
# Load the datasets 
INTERIM_PATH = Path("../data/interim")
PROCESSED_PATH = Path("../data/processed")

monthly_returns = pd.read_parquet(INTERIM_PATH / 'returns_monthly_stocks.parquet', 
                       engine='pyarrow')
monthly_returns

,date,ticker,bench,l_stock,l_bench,ex_log,close,volume
0,2005-02-28,AAPL,^GSPC,0.154188,0.018727,0.135461,1.346702,21448946400
1,2005-03-31,AAPL,^GSPC,-0.073765,-0.019303,-0.054462,1.250938,14675920000
2,2005-04-30,AAPL,^GSPC,-0.144597,-0.020314,-0.124284,1.082525,19375518400
3,2005-05-31,AAPL,^GSPC,0.097677,0.029512,0.068165,1.193600,12858294400
4,2005-06-30,AAPL,^GSPC,-0.077092,-0.000143,-0.076949,1.105041,11327195600
...,...,...,...,...,...,...,...,...
2475,2025-05-31,XOM,^GSPC,-0.032033,0.059705,-0.091738,100.431847,327011700
2476,2025-06-30,XOM,^GSPC,0.061536,0.048416,0.013120,106.806129,410054300
2477,2025-07-31,XOM,^GSPC,0.035002,0.021435,0.013567,110.610725,317767900
2478,2025-08-31,XOM,^GSPC,0.023460,0.018887,0.004573,113.236298,327731800


In [8]:
# Perform cumulative sum on  the excess log of each of the tickers:

def forward_excess_target(df,group_col="ticker", date_col="date", value_col="ex_log", horizon=12):
    # Copy the dataframe 
    df2 = df[[date_col, group_col, value_col]].copy()
    
    # Ensure all monthly returns are sorted by ticker and date 
    df2 = df2.sort_values([group_col, date_col])
    
    # Create a cs col to hold the cumulative sum 
    df2['cs'] = df.groupby(group_col, sort= False)[value_col].cumsum()
    
    # Create a future df to compute the next 12 months
    future_cs = df2.groupby(group_col,sort=False)['cs'].shift(-horizon)
                                                            
    # New col for excess forward log 
    df2["excess_log_fwd12"] = future_cs - df2["cs"]
    df2["excess_simple_fwd12"] = np.expm1(df2["excess_log_fwd12"])

    # Drop rows without a full future window (last h per ticker)
    df2 = df2[df2["excess_log_fwd12"].notna()].reset_index(drop=True)

    return df2


stocks_monthly_targets_fwd12= forward_excess_target(monthly_returns)

stocks_monthly_targets_fwd12

,date,ticker,ex_log,cs,excess_log_fwd12,excess_simple_fwd12
0,2005-02-28,AAPL,0.135461,0.135461,0.361082,0.434882
1,2005-03-31,AAPL,-0.054462,0.080999,0.316503,0.372320
2,2005-04-30,AAPL,-0.124284,-0.043285,0.544076,0.723015
3,2005-05-31,AAPL,0.068165,0.024880,0.343768,0.410252
4,2005-06-30,AAPL,-0.076949,-0.052069,0.377903,0.459222
...,...,...,...,...,...,...
2355,2024-05-31,XOM,-0.055480,-0.036413,-0.216765,-0.194881
2356,2024-06-30,XOM,-0.044411,-0.080825,-0.159233,-0.147202
2357,2024-07-31,XOM,0.018439,-0.062385,-0.164106,-0.151348
2358,2024-08-31,XOM,-0.028074,-0.090459,-0.131459,-0.123185


In [15]:
# Ensure the date column is the correct format 
is_dt = pd.api.types.is_datetime64_any_dtype(stocks_monthly_targets_fwd12["date"])
if not is_dt:
    stocks_monthly_targets_fwd12["date"] = pd.to_datetime(
        stocks_monthly_targets_fwd12["date"],
        errors = 'raise',
        utc = False
    )
else:
    print('Dates are in correct format')

# Check if there are any missing dates
missing = stocks_monthly_targets_fwd12["date"].isna().sum()
if missing:
    print(f"Warning: {missing} NaT in date column")
else:
    print("There are no missing dates")

# Confirm range
print(
    "Date range:",
    stocks_monthly_targets_fwd12["date"].min(),
    "→",
    stocks_monthly_targets_fwd12["date"].max()
)

Dates are in correct format
There are no missing dates
Date range: 2005-02-28 00:00:00 → 2024-09-30 00:00:00


In [36]:
# Add a label to say if the stock beat or did not beat the market that month:

stocks_monthly_targets_fwd12["excess_log_fwd12"] = pd.to_numeric(stocks_monthly_targets_fwd12["excess_log_fwd12"], errors="coerce")
stocks_monthly_targets_fwd12["label_win_12"] = (stocks_monthly_targets_fwd12["excess_log_fwd12"] > 0).astype(int)

stocks_monthly_targets_fwd12.head()

,date,ticker,ex_log,cs,excess_log_fwd12,excess_simple_fwd12,label_win_12
0,2005-02-28,AAPL,0.135461,0.135461,0.361082,0.434882,1
1,2005-03-31,AAPL,-0.054462,0.080999,0.316503,0.372320,1
2,2005-04-30,AAPL,-0.124284,-0.043285,0.544076,0.723015,1
3,2005-05-31,AAPL,0.068165,0.024880,0.343768,0.410252,1
4,2005-06-30,AAPL,-0.076949,-0.052069,0.377903,0.459222,1


In [37]:
# Split data into train, validation, and test 
train_end = pd.Timestamp("2017-12-31")
val_start, val_end = pd.Timestamp("2018-01-01"), pd.Timestamp("2021-12-31")
test_start, test_end = pd.Timestamp("2022-01-01"),pd.Timestamp("2024-09-30")

df= stocks_monthly_targets_fwd12.copy()
df['split'] = np.select(
    [
        df['date'] <= train_end, 
        (df['date'] >= val_start) & (df['date'] <= val_end),
        (df['date'] >= test_start) & (df['date']<= test_end)
    ],
    ['train', 'val', 'test'],
    default = 'drop'
)

df = df[df["split"] != "drop"].reset_index(drop=True)

# Quality checks

In [38]:
# Declared boundaries (feature-time t)
train_end = pd.Timestamp("2017-12-31")
val_start, val_end   = pd.Timestamp("2018-01-01"), pd.Timestamp("2021-12-31")
test_start, test_end = pd.Timestamp("2022-01-01"), pd.Timestamp("2024-09-30")

mm = df.groupby("split")["date"].agg(["min","max"])
print(mm)

# Assert ranges (robust to month-ends)
assert mm.loc["train", "max"] <= train_end

assert val_start  <= mm.loc["val",  "min"] <= val_end
assert val_start  <= mm.loc["val",  "max"] <= val_end

assert test_start <= mm.loc["test", "min"] <= test_end
assert test_start <= mm.loc["test", "max"] <= test_end

print(df["excess_log_fwd12"].isna().sum())

             min        max
split                      
test  2022-01-31 2024-09-30
train 2005-02-28 2017-12-31
val   2018-01-31 2021-12-31
0


In [39]:
print("=== Saving Processed Data ===")

# Ensure processed directory exists
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

# Save the complete dataset with splits
df.to_parquet(PROCESSED_PATH / 'stocks_monthly_targets_splits.parquet', index=False)
print(f"✓ Saved complete dataset: {len(df)} rows")

# Also save separate files for each split
train_df = df[df['split'] == 'train']
val_df = df[df['split'] == 'val']
test_df = df[df['split'] == 'test']

train_df.to_parquet(PROCESSED_PATH / 'train_set.parquet', index=False)
val_df.to_parquet(PROCESSED_PATH / 'val_set.parquet', index=False)
test_df.to_parquet(PROCESSED_PATH / 'test_set.parquet', index=False)

print(f"✓ Train set: {len(train_df)} rows")
print(f"✓ Val set: {len(val_df)} rows")
print(f"✓ Test set: {len(test_df)} rows")
print(f"\nAll data saved to: {PROCESSED_PATH}")

=== Saving Processed Data ===
✓ Saved complete dataset: 2360 rows
✓ Train set: 1550 rows
✓ Val set: 480 rows
✓ Test set: 330 rows

All data saved to: ../data/processed
